# 復電資料前處理

# 1. 資料讀取

In [122]:
import pandas as pd
import os

DATA_DIR = "raw_data"

# 載入合併後的原始資料
df = pd.read_csv(os.path.join(DATA_DIR, "bpa_data_merged.csv"), low_memory=False)

# Check
df.head()

,Out Datetime,In Datetime,Name,Voltage (kV),Line Type,Gen Flag,Length (miles),Duration (minutes),Outage Type,Dispatcher Cause,...,Cause Dispatch,Cause Field,Responsible System Dispatch,Responsible System Field,Cause,Responsible System,O&M District,Transmission Owner NERC TADS,Out Datetime (PPT),In Datetime (PPT)
0,01/01/1988 11:00,NaN,Ashe-Marion No 2 500kV line,500.0,L,T,224.01,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01/01/1988 11:00,NaN,Slatt tap to Ashe-Marion No 2 500kV line,500.0,T,,0.10,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01/20/1995 10:31,NaN,Satsop-Grays Harbor Public Development Authori...,230.0,L,,0.10,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04/03/1995 10:00,NaN,Bell-AVA Beacon No 2 115kV line,115.0,L,,6.20,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,06/13/1996 10:20,NaN,Alvey-Fairview No 1 230kV line,230.0,L,T,97.46,still out,Plan,Normally Out,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 2. 資料檢視

## 特徵欄位項目

In [123]:
# 各年份欄位一覽
for year, group in df.groupby("year"):
    cols = [c for c in group.dropna(axis=1, how="all").columns]
    print(f"{year}: {cols}")

2012: ['Out Datetime', 'In Datetime', 'Name', 'Voltage (kV)', 'Line Type', 'Gen Flag', 'Length (miles)', 'Duration (minutes)', 'Outage Type', 'Dispatcher Cause', 'Field Cause', 'District', 'Outage ID', 'year']
2013: ['Out Datetime', 'In Datetime', 'Name', 'Voltage (kV)', 'Line Type', 'Gen Flag', 'Length (miles)', 'Duration (minutes)', 'Outage Type', 'Dispatcher Cause', 'Field Cause', 'District', 'Outage ID', 'year']
2014: ['Out Datetime', 'In Datetime', 'Name', 'Voltage (kV)', 'Line Type', 'Gen Flag', 'Length (miles)', 'Duration (minutes)', 'Outage Type', 'District', 'Outage ID', 'year', 'Cause Dispatch', 'Cause Field', 'Responsible System Dispatch', 'Responsible System Field']
2015: ['Out Datetime', 'In Datetime', 'Name', 'Voltage (kV)', 'Line Type', 'Gen Flag', 'Length (miles)', 'Duration (minutes)', 'Outage Type', 'District', 'Outage ID', 'year', 'Cause Dispatch', 'Cause Field', 'Responsible System Dispatch', 'Responsible System Field']
2016: ['Out Datetime', 'In Datetime', 'Name', 

In [124]:
print(df.columns.tolist())

['Out Datetime', 'In Datetime', 'Name', 'Voltage (kV)', 'Line Type', 'Gen Flag', 'Length (miles)', 'Duration (minutes)', 'Outage Type', 'Dispatcher Cause', 'Field Cause', 'District', 'Outage ID', 'year', 'Cause Dispatch', 'Cause Field', 'Responsible System Dispatch', 'Responsible System Field', 'Cause', 'Responsible System', 'O&M District', 'Transmission Owner NERC TADS', 'Out Datetime (PPT)', 'In Datetime (PPT)']


## Duplicated Data

In [125]:
f"重複值 : {df.duplicated().sum()}"

'重複值 : 0'

# 3. Feature Selection

* 目標：將各年份欄位名稱不一致的欄位統一，並捨棄不需要的欄位

* 處理項目

    **A. Out Datetime / In Datetime 統一（2022~2023）**

    2022~2023 欄位名稱改為 `Out Datetime (PPT)` / `In Datetime (PPT)`，合併回 `Out Datetime` / `In Datetime`

    **B. Cause 欄合併**

    依優先順序合併，統一欄名為 `Cause`

    * `Cause Field` / `Field Cause` : 現場判斷（優先）
    * `Cause Dispatch` / `Dispatcher Cause` : 調度員判斷（fallback）

    **C. 捨棄不需要欄位**
    最終保留：`Out Datetime`、`In Datetime`、`Name`、`Voltage (kV)`、`Duration (minutes)`、`Outage Type`、`Cause`、`year`

* 特徵欄位說明

    | 欄位名稱 | 說明 | 保留 |
    |---|---|---|
    | `Out Datetime` | 停電開始時間 | ✓ (用以判斷停電年份)|
    | `In Datetime` | 復電時間 | ✓ (用以判斷停電區間)|
    | `Name` | 輸電線路名稱 | ✓ |
    | `Voltage (kV)` | 線路電壓（kV） | ✓ |
    | `Line Type` | 線路類型 | ✗ (用於識別主線路或分支線，不需此資料)|
    | `Gen Flag` | 發電旗標 | ✗ |
    | `Length (miles)` | 線路長度（英里） | ✗ |
    | `Duration (minutes)` | 停電持續時間（分鐘） | ✓ |
    | `Outage Type` | 停電類型（Auto / Plan 等） | ✓ (計畫性停電並非意外造成，需保留本欄以便後續篩選) |
    | `Dispatcher Cause` | 調度員判斷原因（2012–2013） | ✗（合併至 Cause）|
    | `Field Cause` | 現場確認原因（2012–2013） | ✗（合併至 Cause）|
    | `Cause Dispatch` | 調度員判斷原因（2014–2015） | ✗（合併至 Cause）|
    | `Cause Field` | 現場確認原因（2014–2015） | ✗（合併至 Cause）|
    | `Responsible System Dispatch` | 調度員判斷責任系統（2014–2015） | ✗ |
    | `Responsible System Field` | 現場確認責任系統（2014–2015） | ✗ |
    | `District` | 地區（2014–2017） | ✗ |
    | `Outage ID` | 停電事件編號 | ✗ |
    | `year` | 資料年份 | ✓ |
    | `Cause` | 停電原因（2016+） | ✓ |
    | `Responsible System` | 責任系統（2016+） | ✗ |
    | `O&M District` | 地區（2018+） | ✗ |
    | `Transmission Owner NERC TADS` | 輸電業者（NERC TADS 格式） | ✗ |
    | `Out Datetime (PPT)` | 停電開始時間，PPT 時區（2022~2023） | ✗（合併至 Out Datetime）|
    | `In Datetime (PPT)` | 復電時間，PPT 時區（2022~2023） | ✗（合併至 In Datetime）|

In [126]:
# Out Datetime / In Datetime 統一（2022~2023 欄位名稱不同）
df["Out Datetime"] = df["Out Datetime"].fillna(df["Out Datetime (PPT)"])
df["In Datetime"] = df["In Datetime"].fillna(df["In Datetime (PPT)"])

# Cause 欄合併：Cause Field 優先，依序 fallback
df["Cause"] = (df["Cause"]
               .fillna(df["Cause Field"])
               .fillna(df["Field Cause"])
               .fillna(df["Cause Dispatch"])
               .fillna(df["Dispatcher Cause"]))

# 保留分析所需欄位
keep_cols = ["Name", "Out Datetime", "In Datetime", "Voltage (kV)", "Duration (minutes)", "Outage Type", "Cause", "year"]
df = df[keep_cols]

# Check
df

,Name,Out Datetime,In Datetime,Voltage (kV),Duration (minutes),Outage Type,Cause,year
0,Ashe-Marion No 2 500kV line,01/01/1988 11:00,NaN,500.0,still out,Plan,Normally Out,2012
1,Slatt tap to Ashe-Marion No 2 500kV line,01/01/1988 11:00,NaN,500.0,still out,Plan,Normally Out,2012
2,Satsop-Grays Harbor Public Development Authori...,01/20/1995 10:31,NaN,230.0,still out,Plan,Normally Out,2012
3,Bell-AVA Beacon No 2 115kV line,04/03/1995 10:00,NaN,115.0,still out,Plan,Normally Out,2012
4,Alvey-Fairview No 1 230kV line,06/13/1996 10:20,NaN,230.0,still out,Plan,Normally Out,2012
...,...,...,...,...,...,...,...,...
55083,Yaak-Moyie Springs section of Libby-Bonners Fe...,12/30/2023 07:11,12/30/2023 07:11,115.0,0,Auto,Unknown,2023
55084,Ashe-Slatt No 1 500kV line,12/31/2023 03:12,12/31/2023 03:12,500.0,0,Auto,Weather,2023
55085,Ashe-Slatt No 1 500kV line,12/31/2023 03:33,12/31/2023 03:33,500.0,0,Auto,Weather,2023
55086,Ashe-Slatt No 1 500kV line,12/31/2023 03:39,12/31/2023 03:39,500.0,0,Auto,Weather,2023


# 4. 檢視資料內容

In [127]:
# Outage Type 值統計
print("Outage Type:")
print(df["Outage Type"].value_counts())

# Cause 值統計
print("\nCause:")
print(df["Cause"].value_counts())

Outage Type:
Outage Type
Plan    30526
Auto    24562
Name: count, dtype: int64

Cause:
Cause
Maintenance              17444
Unknown                   7135
Lightning                 5474
Construction              2484
Voltage Control           2240
                         ...  
Overload                     5
Line or Bank charging        4
Industrial                   2
Out of step                  2
Voltage                      1
Name: count, Length: 69, dtype: int64


## 統計因素原因

In [128]:
# 印出所有 Cause 值
print(df["Cause"].value_counts().to_string())

Cause
Maintenance                   17444
Unknown                        7135
Lightning                      5474
Construction                   2484
Voltage Control                2240
Weather                        2016
Normally Out                   1795
Switching                      1690
Emergency                      1222
Foreign Request                1092
Foreign Trouble                1092
Tree blown                      901
Wind                            777
Urgent                          754
Forced (Configuration)          677
Line Material Failure           628
Ice                             588
Fire                            568
Terminal Equipment Failure      534
Oper Plan/RTCA Reqd Action      530
Proximity/Other                 353
Bird or Animal                  343
Forced                          319
Tree                            306
Bird droppings                  302
Contamination                   291
Load Control                    290
Maintenance - TOp     

# 5. Data Cleaning

## 目標
從合併後的資料中，篩選出適合分析的停電事件

## 處理步驟

**A. 篩選自動停電（Auto）**

僅保留非計畫性的自動停電事件，排除計畫性維修停電（`Plan`）

**B. 排除 still out**

`Duration (minutes)` 為 `still out` 表示該停電事件在資料截止日仍未復電，無法取得完整復電時間，予以排除

**C. Duration 轉數值**

將 `Duration (minutes)` 欄轉為數值型態（`float`），非數值內容轉為 `NaN`

**D. 排除瞬間停電（duration ≤ 0）**

持續時間為 0 或負值者視為瞬間停電，不納入分析

**E. 排除 Duration 為 NaN**

轉型後仍為 `NaN` 者直接排除該筆資料

**F. Cause 標準化**

將原始 Cause 值對應至以下六個類別，其餘歸為 `Other`

| 標準類別 | 對應原始值 |
|---|---|
| `Foreign Trouble` | `Foreign Object`, `Vehicle`, `Aircraft`, `Machinery, Logging`, `Machinery, Construction`, `Machinery, Farming`, `Bird or Animal`, `Bird droppings`, `Bird Droppings` |
| `Unknown` | `Unknown`, `Not Reported` |
| `Lightning` | `Lightning` |
| `Tree Blown` | `Tree blown`, `Tree Blown`, `Tree`, `Tree cut`, `Tree Cut` |
| `Wind` | `Wind`, `Galloping Conductors` |
| `Weather` | `Weather`, `Ice`, `Dust`, `Smoke`, `Contamination`, `Fire` |
| `Other` | 其餘所有(如 : 人為因素、山崩、施工意外...等) |

In [129]:
# A. 篩選自動停電（Auto）
df = df[df["Outage Type"] == "Auto"].copy()

# B. 排除 still out
df = df[df["Duration (minutes)"] != "still out"]

# C. Duration 轉數值
# "coerce" : 遭於無法解析的值，使用 NaN 寫入
df["Duration (minutes)"] = pd.to_numeric(df["Duration (minutes)"], errors="coerce")

# D. 排除瞬間停電（duration ≤ 0）
df = df[df["Duration (minutes)"] > 0]

# E. 排除 Duration 為 NaN
df = df[df["Duration (minutes)"].notna()]

# F. Cause 標準化
CAUSE_MAP = {
    "Foreign Trouble": ["Foreign Object", "Vehicle", "Aircraft", "Machinery, Logging",
                        "Machinery, Construction", "Machinery, Farming", "Bird or Animal",
                        "Bird droppings", "Bird Droppings"],
    "Unknown":         ["Unknown", "Not Reported"],
    "Lightning":       ["Lightning"],
    "Tree Blown":      ["Tree blown", "Tree Blown", "Tree", "Tree cut", "Tree Cut"],
    "Wind":            ["Wind", "Galloping Conductors"],
    "Weather":         ["Weather", "Ice", "Dust", "Smoke", "Contamination", "Fire"],
}

# 將 CAUSE_MAP 反轉為 value -> key 的對應
cause_lookup = {v: k for k, vs in CAUSE_MAP.items() for v in vs}

# 對應標準類別，未對應者歸為 Other
df["Cause"] = df["Cause"].map(cause_lookup).fillna("Other")

# Check
df["Cause"].value_counts()

Cause
Other              3182
Unknown            1605
Weather            1539
Tree Blown         1301
Lightning           973
Wind                547
Foreign Trouble     364
Name: count, dtype: int64

# 6. Feature Creation 新增停電事件跨越年份

In [130]:
# 解析時間欄位
df["Out Datetime"] = pd.to_datetime(df["Out Datetime"], errors="coerce")
df["In Datetime"]  = pd.to_datetime(df["In Datetime"],  errors="coerce")

# 產生停電事件經過的年份 set
def get_outage_years(row):
    out = row["Out Datetime"]
    inp = row["In Datetime"]
    # 不處理停電事件時間記錄為空值者
    if pd.isna(out):
        return None
    # 復電時間為空值，僅記錄停電時間年份
    if pd.isna(inp):
        return {int(out.year)}
    return set(range(int(out.year), int(inp.year) + 1))

df["outage_year"] = df.apply(get_outage_years, axis=1)

# Check：找出跨年事件
cross_year = df[df["outage_year"].apply(lambda x: len(x) > 1 if not pd.isna(x) else False)]
print(f"跨年事件數：{len(cross_year)} 筆")
cross_year[["Out Datetime", "In Datetime", "outage_year"]].head(10)

跨年事件數：26 筆


,Out Datetime,In Datetime,outage_year
5454,2012-12-17 05:31:00,2013-01-10 11:42:00,"{2012, 2013}"
5455,2012-12-17 05:31:00,2013-01-10 11:42:00,"{2012, 2013}"
26783,2016-12-14 23:52:00,2017-02-02 15:51:00,"{2016, 2017}"
26784,2016-12-14 23:52:00,2017-02-02 15:51:00,"{2016, 2017}"
31886,2017-11-16 10:27:00,2018-04-03 09:50:00,"{2017, 2018}"
31887,2017-11-16 10:27:00,2018-04-03 09:50:00,"{2017, 2018}"
41500,2019-12-16 08:26:00,2020-01-09 13:27:00,"{2019, 2020}"
41501,2019-12-16 08:26:00,2020-01-09 13:27:00,"{2019, 2020}"
41506,2019-12-31 22:37:00,2020-01-01 06:35:00,"{2019, 2020}"
41507,2019-12-31 22:37:00,2020-01-01 06:35:00,"{2019, 2020}"


#  7. Check Missing Data

In [131]:
print("缺失值：")
print(df.isnull().sum())

缺失值：
Name                  0
Out Datetime          0
In Datetime           0
Voltage (kV)          0
Duration (minutes)    0
Outage Type           0
Cause                 0
year                  0
outage_year           0
dtype: int64


# 異常值處理

* 使用 DOE-417 歷史資料中，所有年份的最長復電時間作為篩選 **異常值** 之 **閥值**

* 異常值處理並非要處理掉極端搶修時間的案件，完全同意有極端修復時間的案例，但在做資料分析時，發現有部分值大於 3 個月，這是不可能出現在搶修下的情況，判斷為電力公司採用其他方式復電，而非實際強修時間，故予以排除

* 當事件跨越年度時，會採用跨越年度的修復時間最大值作為 **閥值**

In [ ]:
# 讀取 DOE-417 資料
df_doe = pd.read_csv("data/doe_wecc.csv")
# 依據年份選出該年度最長的修復時間作為閥值
df_threshold = df_doe.groupby("year")["duration_minutes"].max().reset_index()

# 建立年份 → max 閾值的 dict
threshold_dict = df_threshold.set_index("year")["duration_minutes"].to_dict()

# 全局最大值（找不到年份時的保底）
global_max = max(threshold_dict.values())

# 對每筆事件，取 outage_year set 中所有年份的 max 閾值
def get_threshold(outage_years):
    if pd.isna(outage_years):
        return global_max
    values = [threshold_dict.get(y) for y in outage_years if threshold_dict.get(y) is not None]
    return max(values) if values else global_max

df["threshold"] = df["outage_year"].apply(get_threshold)

# 篩選
before = len(df)
# 取出小於 閥值 的資料
df = df[df["Duration (minutes)"] <= df["threshold"]].copy()
after = len(df)
print(f"移除異常值後: {before} → {after}，移除 {before - after} 筆")

移除異常值後: 9511 → 9356，移除 155 筆


# 存檔

In [134]:
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)
df.to_csv(os.path.join(output_dir, "df_clean.csv"), index=False)